**Desenvolvimento de IA para Análise Preditiva [T3] - M1S08 - Mini-Projeto Avaliativo_17082026**

Aluna: Isabela Gomes da Costa

Desafio proposto:
Você acaba de ser contratado como Analista de Dados Júnior em uma empresa de
varejo que possui um histórico de vendas em formato CSV e precisa de um relatório analítico
para a reunião trimestral da diretoria. O time de negócios quer entender:


*   Como as vendas se comportam ao longo do tempo (por mês e por trimestre);
*   Quais produtos e categorias geram mais receita?
*   Quais regiões têm melhor desempenho?
*   Quais clientes são mais valiosos (segmentação simples por faixa de gasto)?
*   Existe relação entre a quantidade vendida e a receita gerada por transação?

1 - Ambiente e importações

In [1]:
# Bibliotecas de terceiros
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos da biblioteca padrão do Python
import os
import re
import json
import random
from datetime import datetime, timedelta

""" Pasta onde os resultados desta aula serão salvos
PASTA_SAIDA = "saida_aula"
os.makedirs(f"{PASTA_SAIDA}/graficos", exist_ok=True)
print("Pasta de saída criada:", PASTA_SAIDA)"""

print("Pandas:", pd.__version__)
print("NumPy :", np.__version__)

Pandas: 2.2.3
NumPy : 2.0.2


2 - Criarção do Dataset de Vendas

In [13]:
def gerar_dataset_vendas(n_registros=500, seed=42):
    # Gera um dataset sintético de vendas com dados sujos.

    random.seed(seed)
    np.random.seed(seed)

    produtos = [
        "Camiseta",
        "Calca Jeans",
        "Vestido",
        "Blusa",
        "Jaqueta",
        "Shorts",
        "Saia",
        "Moletom",
        "Camisa Social",
        "Tenis"
    ]

    categorias = {
        "Camiseta": "Feminino e Masculino",
        "Calca Jeans": "Feminino e Masculino",
        "Vestido": "Feminino",
        "Blusa": "Feminino",
        "Jaqueta": "Feminino e Masculino",
        "Shorts": "Feminino e Masculino",
        "Saia": "Feminino",
        "Moletom": "Feminino e Masculino",
        "Camisa Social": "Masculino",
        "Tenis": "Calcados"
    }

    precos = {
        "Camiseta": 79.90,
        "Calca Jeans": 189.90,
        "Vestido": 159.90,
        "Blusa": 99.90,
        "Jaqueta": 249.90,
        "Shorts": 89.90,
        "Saia": 119.90,
        "Moletom": 179.90,
        "Camisa Social": 169.90,
        "Tenis": 299.90
    }

    regioes = [
        "Sudeste",
        "Sul",
        "Nordeste",
        "Centro-Oeste",
        "Norte"
    ]

    data_inicio = datetime(2025, 1, 1)

    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)

        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")

        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---

        if random.random() < 0.05:
            quantidade = None  # valor nulo

        if random.random() < 0.04:
            preco = None  # valor nulo

        if random.random() < 0.06:
            produto = " " + produto + " "  # espaços extras

        if random.random() < 0.03:
            data_txt = "DATA INVALIDA"  # data inválida

        if random.random() < 0.10:
            cliente = random.choice([
                cliente.upper().replace("_", "-"),
                cliente + "!!",
                " " + cliente,
                cliente.replace("Cliente_", "cliente#"),
            ])

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()

df_bruto.to_csv("vendas.csv", index=False)

print(f"Dataset gerado com {len(df_bruto)} registros.\n")
print(df_bruto.head())

Dataset gerado com 500 registros.

   id_venda  data_venda      cliente      produto             categoria  \
0         1  2025-05-06  Cliente_015  Calca Jeans  Feminino e Masculino   
1         2  2025-09-16  Cliente_039     Camiseta  Feminino e Masculino   
2         3  2025-03-23  Cliente_045      Jaqueta  Feminino e Masculino   
3         4  2025-11-06  Cliente_017  Calca Jeans  Feminino e Masculino   
4         5  2025-07-05  Cliente_037      Jaqueta  Feminino e Masculino   

         regiao  quantidade  preco_unitario  
0       Sudeste         1.0          203.66  
1         Norte         NaN           73.16  
2  Centro-Oeste         1.0          269.30  
3         Norte         6.0          209.70  
4           Sul        10.0          278.80  


Foi considerado o exemplo de criação de dataset sintético, no entanto, modifiquei para artigos de vestuário e aumentei a quantidade de registros para 500.

3 - Inspeção e Descrição dos dados

In [18]:
df = df_bruto.copy()

def inspecionar_dados(df):
    """Exibe as informacoes estruturais do DataFrame."""
    print("\n=== INSPECAO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df

inspecionar_dados(df)


=== INSPECAO INICIAL DO DATASET ===
Shape: (500, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade        29
preco_unitario    16
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente      produto             categoria  \
0         1  2025-05-06  Cliente_015  Calca Jeans  Feminino e Masculino   
1         2  2025-09-16  Cliente_039     Camiseta  Feminino e Masculino   
2         3  2025-03-23  Cliente_045      Jaqueta  Feminino e Masculino   
3         4  2025-11-06  Cliente_017  Calca Jeans  Feminino e Ma

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-06,Cliente_015,Calca Jeans,Feminino e Masculino,Sudeste,1.0,203.66
1,2,2025-09-16,Cliente_039,Camiseta,Feminino e Masculino,Norte,NaN,73.16
2,3,2025-03-23,Cliente_045,Jaqueta,Feminino e Masculino,Centro-Oeste,1.0,269.30
3,4,2025-11-06,Cliente_017,Calca Jeans,Feminino e Masculino,Norte,6.0,209.70
4,5,2025-07-05,Cliente_037,Jaqueta,Feminino e Masculino,Sul,10.0,278.80
...,...,...,...,...,...,...,...,...
495,496,2025-02-27,Cliente_012,Moletom,Feminino e Masculino,Centro-Oeste,2.0,191.92
496,497,2025-12-31,Cliente_013,Camiseta,Feminino e Masculino,Sul,3.0,78.79
497,498,2025-08-19,Cliente_041,Calca Jeans,Feminino e Masculino,Sul,2.0,167.92
498,499,2025-03-29,Cliente_022,Saia,Feminino,Sudeste,9.0,115.61
